## Importação de Bibliotecas

In [ ]:
# Dependências
import sys
#!{sys.executable} -m pip install --disable-pip-version-check -r ../requirements.txt -q
print('Bibliotecas instaladas')

In [ ]:
# Acesso aos módulos do diretório
from pathlib import Path
PROJECT_ROOT = Path().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

# Manipulação dos dados 
import pandas as pd
import numpy as np
import pickle

# Visualização dos dados
import matplotlib.pyplot as plt
import seaborn as sns

# Funções customizadas
from configs.paths import *
from configs.function_basic import *
from configs.function_others import *

# Avisos
import warnings
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')

# Configuração
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)

print('✅ Bibliotecas carregadas com sucesso')

### Parâmetros globais

In [ ]:
# parâmetros globais
# define a coluna alvo do modelo
TARGET = 'FPD'
# garante reprodutibilidade dos experimentos
RANDOM_STATE = 42
# percentual máximo de valores ausentes permitido para manter a variável
PERCENTUAL_MAX_FALTANTES  = 70

## Carregamento dos Dados

In [ ]:
# Carregar dados brutos
abt00 = pd.read_parquet(RAW_DIR / 'base_tabelao.parquet')

print(f'✅ Dados carregados: {abt00.shape[0]:,} linhas e {abt00.shape[1]:,} colunas')
print(f'\nTarget (FPD) distribuição:')
print((abt00['FPD'].value_counts(normalize=True) * 100).round(2))

In [ ]:
# Nome do arquivo
ARTIFACT_NAME_INFO = 'metadados.csv'

# Carregar os dados 
metadados = pd.read_csv( ARTIFACT_DIR / ARTIFACT_NAME_INFO)

## Grupo Controle

In [ ]:
# cria flag para identificar clientes do grupo controle (CPF 6º e 7º dígitos = ZZ ou ZX)
abt00['FLAG_GRUPO_CONTROLE'] = (abt00['NUM_CPF'].astype(str).str[5:7].isin(['ZZ', 'ZX']).astype(int))

# checar distribuição da flag
abt00['FLAG_GRUPO_CONTROLE'].value_counts(normalize=True) * 100

## Definir filtro grupo controle

In [ ]:
# controlar se aplica ou não o filtro do grupo controle
APLICAR_FILTRO_PADRAO = True  # True = sem grupo controle | False = base completa

if APLICAR_FILTRO_PADRAO:
    abt01 = abt00[abt00['FLAG_GRUPO_CONTROLE'] == 0].copy()
    abt01.drop(columns=['FLAG_GRUPO_CONTROLE'], inplace=True)
else:
    abt01 = abt00.copy()

print(f"Modo ativo: {'Sem grupo controle' if APLICAR_FILTRO_PADRAO else 'base completa'}")
print(f"Base ativa: {len(abt01):,} registros")

## Divisão Treino/Teste (Temporal por SAFRA)

In [ ]:
# Verificar SAFRAs disponíveis
if 'SAFRA' in abt01.columns:
    print(f'\n📅 SAFRAs disponíveis:')
    print(abt01['SAFRA'].value_counts().sort_index())

### Separação dos dados para validação temporal (Out-of-Time)

A separação dos dados é realizada com base na **safra**, respeitando a ordem temporal das observações.  
Essa abordagem, conhecida como **validação Out-of-Time (OOT)**, evita vazamento de informação e simula o comportamento real do modelo em dados futuros.

In [ ]:
# garante SAFRA como inteiro
abt01['SAFRA'] = abt01['SAFRA'].astype(int)
safra_counts = abt01['SAFRA'].value_counts().sort_index()

# Definir SAFRAs de teste (Fevereiro e Março 2025)
test_safras = [202502, 202503]

# Criar máscaras
test_mask = abt01['SAFRA'].isin(test_safras)
train_mask = ~test_mask

# Separar dados
df_train = abt01[train_mask].copy()
df_test = abt01[test_mask].copy()

## Backup e Análise Inicial

In [ ]:
# Backup dos dados originais
abt02 = df_train.copy()

print(f'Shape original: {abt02.shape}')

## Tratamento para modelagem

In [ ]:
# ajuste de tipagem da data de nascimento
abt02['DATADENASCIMENTO'] = pd.to_datetime(abt02['DATADENASCIMENTO'], format='%d/%m/%Y', errors='coerce')

## Feature Engineering

In [ ]:
# calcula idade do cliente na safra
abt02['DATADENASCIMENTO'] = pd.to_datetime(abt02['DATADENASCIMENTO'], dayfirst=True)
abt02['SAFRA_DT'] = pd.to_datetime(abt02['SAFRA'].astype(str) + '01', format='%Y%m%d')
abt02['IDADE'] = ((abt02['SAFRA_DT'] - abt02['DATADENASCIMENTO']).dt.days // 365)

In [ ]:
# remove variáveis que causam vazamento, identificação ou controle temporal
ignore_cols = ['SAFRA', 'FPD', 'FPD_bureau', 'FPD_telco', 'flag_mig2', 'flag_mig2_bureau', 'flag_mig2_telco', 'NUM_CPF', 'DATADENASCIMENTO', 'SAFRA_DT']

abt02 = abt02.drop(columns=ignore_cols, errors='ignore')

>As variáveis removidas incluem identificadores únicos, variáveis de controle temporal e indicadores diretamente relacionados ao evento de inadimplência, prevenindo vazamento de informação >e garantindo aderência ao cenário real de decisão de crédito.

## Remoção de Colunas Desnecessárias

In [ ]:
# filtra variáveis com muitos nulos OU cardinalidade igual a 1
df_drop = metadados[(metadados['PC_nulos'] >= PERCENTUAL_MAX_FALTANTES) | (metadados['Cardinalidade'] <= 1)]
var_drop = list(df_drop.Feature.values)

# efeito real do drop
qtd_excluir = abt02.columns.isin(var_drop).sum()
print(qtd_excluir)

# remover variáveis do ABT ignorando colunas inexistentes
abt02 = abt02.drop(columns=var_drop, errors='ignore')

print(var_drop)

In [ ]:
# Salvar a lista em um arquivo .pkl
artifact_path = Path(ARTIFACT_DIR) / 'var_drop.pkl'

with open(artifact_path, 'wb') as f:
    pickle.dump(var_drop, f)

In [ ]:
# Valida persistência
# recarregar do pickle e comparar
with open(artifact_path, 'rb') as f:
    var_drop_reload = pickle.load(f)

set(var_drop) == set(var_drop_reload)

## Tratamento de Valores Faltantes

In [ ]:
# Análise de missing values restantes
abt02, stats = custom_fillna(abt02, strategy='median')

In [ ]:
# Salvar a lista em um arquivo .pkl
artifact_path = Path(ARTIFACT_DIR) / 'stats_nulo.pkl'

with open(artifact_path, 'wb') as f:
    pickle.dump(stats, f)

In [ ]:
# Valida persistência
# reload e comparação
with open(artifact_path, 'rb') as f:
    stats_reload = pickle.load(f)

stats == stats_reload

## Cardinalidade

### Coluna STATUSRF

In [ ]:
# binariza STATUSRF: REGULAR = 0, demais = 1 e converte para category
abt02['STATUSRF'] = (abt02['STATUSRF'] != 'REGULAR').astype('int8').astype('category')

### Coluna CEP_3_digitos

In [ ]:
# consolida CEP_3_digitos em macro-região para reduzir cardinalidade e capturar sinal geográfico
abt02['CEP_REGIAO'] = abt02['CEP_3_digitos'].apply(cep_3_para_regiao)

# remove a coluna original após a consolidação
abt02.drop(columns=['CEP_3_digitos'], inplace=True)

In [ ]:
# one-hot encoding da variável CEP_REGIAO para uso em regressão logística
abt02 = pd.get_dummies(abt02,columns=['CEP_REGIAO'],prefix='CEP', dtype=int, drop_first=True) # evita multicolinearidade (dummy trap)

In [ ]:
# congela colunas de CEP para uso no score
CEP_DUMMY_COLS = [c for c in abt02.columns if c.startswith('CEP_')]

In [ ]:
# salvar contrato do one-hot (colunas esperadas no score)
artifact_path_onehot = Path(ARTIFACT_DIR) / 'cep_onehot_cols.pkl'

with open(artifact_path_onehot, 'wb') as f:
    pickle.dump(CEP_DUMMY_COLS, f)

## Preparando os dados de Teste

In [ ]:
# função para padronizar tratamento inicial e engenharia básica de features
def preprocess_base(df):
    df = df.copy()

    # tipagem
    df['DATADENASCIMENTO'] = pd.to_datetime(df['DATADENASCIMENTO'], format='%d/%m/%Y', errors='coerce')

    # datas auxiliares
    df['SAFRA_DT'] = pd.to_datetime(df['SAFRA'].astype(str) + '01', format='%Y%m%d', errors='coerce')

    # idade na safra
    df['IDADE'] = ((df['SAFRA_DT'] - df['DATADENASCIMENTO']).dt.days // 365)

    # colunas a remover
    ignore_cols = ['SAFRA', 'FPD', 'FPD_bureau', 'FPD_telco', 'flag_mig2', 'flag_mig2_bureau', 'flag_mig2_telco', 'NUM_CPF', 'DATADENASCIMENTO', 'SAFRA_DT']

    df = df.drop(columns=ignore_cols, errors='ignore')

    return df

In [ ]:
# Backup dos dados originais
abt01_test = df_test.copy()

abt01_test  = preprocess_base(abt01_test)

In [ ]:
# remover variáveis descartadas no treino
with open(Path(ARTIFACT_DIR) / 'var_drop.pkl', 'rb') as f:
    var_drop = pickle.load(f)
    
# aplicar no conjunto de teste
abt01_test.drop(columns=var_drop, inplace=True, errors='ignore')

In [ ]:
# carregar estatísticas de imputação
with open(Path(ARTIFACT_DIR) / 'stats_nulo.pkl', 'rb') as f:
    stats = pickle.load(f)

# imputação numérica
for col, value in stats['numerical'].items():
    if col in abt01_test.columns:
        abt01_test[col] = abt01_test[col].fillna(value)

# imputação categórica (somente colunas existentes no teste)
cat_cols = [c for c in stats['categorical_cols'] if c in abt01_test.columns]
abt01_test[cat_cols] = abt01_test[cat_cols].fillna(stats['categorical_fill'])

In [ ]:
# Aplicar imputação no dataset de teste
with open(Path(ARTIFACT_DIR) / 'stats_nulo.pkl', 'rb') as f:
    stats = pickle.load(f)

# numéricas
for col, value in stats['numerical'].items():
    if col in abt01_test.columns:
        abt01_test[col] = abt01_test[col].fillna(value)

# categóricas (usando contrato do treino)
cat_cols = stats['categorical_cols']
abt01_test[cat_cols] = abt01_test[cat_cols].fillna(stats['categorical_fill'])


In [ ]:
# binariza STATUSRF: REGULAR = 0, demais = 1 e converte para category
abt01_test['STATUSRF'] = (abt01_test['STATUSRF'] != 'REGULAR').astype('int8').astype('category')

In [ ]:
# carregar colunas de CEP one-hot do treino
with open(Path(ARTIFACT_DIR) / 'cep_onehot_cols.pkl', 'rb') as f:
    CEP_DUMMY_COLS = pickle.load(f)

# criar macro-região no teste
abt01_test['CEP_REGIAO'] = abt01_test['CEP_3_digitos'].apply(cep_3_para_regiao)

# remover coluna original
abt01_test.drop(columns=['CEP_3_digitos'], inplace=True)

# one-hot encoding no teste
abt01_test = pd.get_dummies(
    abt01_test,
    columns=['CEP_REGIAO'],
    prefix='CEP',
    dtype=int,
    drop_first=True
)

## Sanity check

In [ ]:
cols_train = set(abt02.columns)
cols_test = set(abt01_test.columns)

print('Só no treino:', cols_train - cols_test)
print('Só no teste:', cols_test - cols_train)

In [ ]:
# Análise de missing values
analyze_missing_values(abt01_test, plot=False)

# Trazer o target para a tabela pós dataprep

In [ ]:
# Checar se número de linhas bate
print("Treino:")
print("linhas Dataset Treino tratados:", len(abt02))
print("linhas Treino:", len(df_train))

print("\nTeste:")
print("linhas Dataset Teste tratados:", len(abt01_test))
print("linhas Teste:", len(df_test))

In [ ]:
# Reanexar target e o controle temporal antes de salvar

# Para treino
abt02_train_final = abt02.copy()
abt02_train_final['SAFRA'] = df_train['SAFRA'] # # adiciona SAFRA para controle temporal do model
abt02_train_final['FPD'] = df_train['FPD']  # reanexa target original

# Para teste
abt01_test_final = abt01_test.copy()
abt01_test_final['SAFRA'] = df_test['SAFRA']
abt01_test_final['FPD'] = df_test['FPD']    # reanexa target original

## Salvamento dos Dados Processados

In [ ]:
# Salvar datasets processados e lista de features

print('\n💾 Salvando dados processados...')

# Dataset completo (opcional)
abt02_train_final.to_parquet(PROCESSED_DIR / 'df_train.parquet', index=False)
abt01_test_final.to_parquet(PROCESSED_DIR / 'df_test.parquet', index=False)
print(f'   ✓ Treino salvo: {PROCESSED_DIR / "df_train.parquet"}')
print(f'   ✓ Teste salvo: {PROCESSED_DIR / "df_test.parquet"}')

# Salvar lista de features (excluindo a target)
features_list = [c for c in abt02.columns if c != 'FPD']
with open(ARTIFACT_DIR / 'features_list.pkl', 'wb') as f:
    pickle.dump(features_list, f)
print(f'   ✓ Lista de features salva em: {ARTIFACT_DIR / "features_list.pkl"}')

print(f'\n✅ Todos os dados processados foram salvos')

## Resumo do Tratamento

In [ ]:
# Resumo do tratamento dos dados

print('\n' + '='*60)
print('RESUMO DO TRATAMENTO DOS DADOS')
print('='*60)

# Shapes - comparando treino original vs final processado
print(f'\n✅ Shape original (treino antes do processamento): {df_train.shape[0]:,} registros × {df_train.shape[1]} colunas')
print(f'✅ Shape final (treino após processamento): {abt02_train_final.shape[0]:,} registros × {abt02_train_final.shape[1]} colunas')

# Colunas removidas e novas features
cols_original = df_train.shape[1]
cols_final = abt02_train_final.shape[1]
new_features = 3  # IDADE, SCORE_MEDIO, SCORE_DIFF
cols_removed = cols_original + new_features - cols_final

print(f'\n✅ Colunas removidas no pipeline: {cols_removed}')
print(f'✅ Novas features criadas: {new_features} (IDADE, SCORE_MEDIO, SCORE_DIFF)')
print(f'✅ Variação líquida: {cols_final - cols_original:+d} colunas')

# Missing values - comparando original vs final
missing_orig = df_train.isnull().sum().sum()
missing_final = abt02_train_final.isnull().sum().sum()
print(f'\n✅ Missing values originais: {missing_orig:,}')
print(f'✅ Missing values finais: {missing_final:,}')
print(f'✅ Redução de missing: {missing_orig - missing_final:,} ({((missing_orig - missing_final) / max(missing_orig, 1) * 100):.1f}%)')

# Divisão temporal por SAFRA
print(f'\n✅ Divisão treino/teste: TEMPORAL por SAFRA')
treino_safras = sorted(df_train["SAFRA"].unique().tolist())
teste_safras = sorted(df_test["SAFRA"].unique().tolist())
print(f'   📊 Treino: SAFRAs {treino_safras} → {abt02_train_final.shape[0]:,} registros')
print(f'   📊 Teste: SAFRAs {teste_safras} → {abt01_test_final.shape[0]:,} registros')

# Distribuição da variável target
print(f'\n✅ Distribuição do target (FPD) no treino:')
fpd_dist = abt02_train_final['FPD'].value_counts(normalize=True).sort_index()
for label, pct in fpd_dist.items():
    print(f'   • Classe {label}: {pct*100:.2f}%')

print(f'\n✅ Dados prontos para modelagem!')
print('='*60)